# GridSearchCV — ElasticNet

Notebook independente: execute as células em ordem. Aqui estão apenas a preparação e a análise do **ElasticNet**. Para comparar os três modelos com as mesmas métricas, abra `02_GridSearchCV.ipynb`.


## 1. Importações e dados

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    PowerTransformer,
    StandardScaler,
    QuantileTransformer,
)

# Localiza Regularização ao executar da raiz do workspace, do projeto ou de notebooks.
PASTA_ATUAL = Path.cwd().resolve()
PASTA_PROJETO = next(
    pasta
    for base in (PASTA_ATUAL, *PASTA_ATUAL.parents)
    for pasta in (base, base / "Regularização")
    if (pasta / "src" / "config.py").is_file()
    and (pasta / "dados" / "diabetes_categorizado.parquet").is_file()
)
if str(PASTA_PROJETO) not in sys.path:
    sys.path.insert(0, str(PASTA_PROJETO))

from src.utils import dataframe_coeficientes
from src.config import DADOS_DIABETES_CATEGORIZADOS
from src.graficos import plot_comparar_metricas_modelos, plot_coeficientes
from src.modelos import (
    RANDOM_STATE,
    grid_search_cv_regressor,
    organiza_resultados,
    construir_pipeline_modelo_regressao,
)

sns.set_theme(palette="bright")

In [ ]:
df = pd.read_parquet(DADOS_DIABETES_CATEGORIZADOS)

df.head()

In [ ]:
df.info()

In [ ]:
X = df.drop(columns="target")
y = df["target"]

## 2. Pré-processamento compartilhado

Os três modelos testam os mesmos três pré-processadores e a mesma opção de transformar o alvo. As transformações ficam dentro do pipeline: são ajustadas apenas nos dados de treino de cada divisão, evitando vazamento de dados.

In [ ]:
colunas_power_transform = ["imc", "ldl", "hdl", "colesterol_total"]

coluna_target = ["target"]

coluna_ordinal_encoder = ["colesterol_hdl_cat"]

coluna_one_hot_encoder = ["sexo"]

colunas_standard_scaler = [
    coluna
    for coluna in df.columns
    if coluna not in colunas_power_transform + coluna_target + coluna_ordinal_encoder + coluna_one_hot_encoder
]

colunas_standard_scaler

In [ ]:
categorias_ordinal_encoder = [
    ["2-3", "4-5", "6+"],
]

In [ ]:
preprocessamento_categoricas = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ],
    remainder="passthrough",
)

preprocessamento_simples = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
        ("standard_scaler", StandardScaler(), X.columns.difference(coluna_ordinal_encoder + coluna_one_hot_encoder)),
    ],
    remainder="passthrough",
)

preprocessamento_completo = ColumnTransformer(
    [
        ("power_transform", PowerTransformer(method="box-cox"), colunas_power_transform),
        ("standard_scaler", StandardScaler(), colunas_standard_scaler),
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ]
)

target_transformer = QuantileTransformer(n_quantiles=20, output_distribution="normal", random_state=RANDOM_STATE)

## 3. Regras comuns das buscas

`alpha` controla a intensidade da regularização. Todos testam os mesmos valores; ElasticNet testa também `l1_ratio`.

A função `grid_search_cv_regressor` usa cinco divisões com embaralhamento e semente fixa, calcula R², MAE e RMSE e escolhe pelo **menor RMSE**. No scikit-learn os erros são armazenados com sinal negativo: `best_score_` é o RMSE negativo. Nas tabelas abaixo, MAE e RMSE aparecem positivos.

As buscas nesta etapa usam todos os dados para explorar configurações e coeficientes. Seus melhores escores não são uma avaliação independente; essa avaliação vem na seção final.

In [ ]:
ALPHAS = [0.01, 0.05, 0.1, 1.0, 5.0, 10.0]
PREPROCESSADORES = [
    preprocessamento_categoricas,
    preprocessamento_simples,
    preprocessamento_completo,
]
TRANSFORMADORES_ALVO = [target_transformer, None]

# As mesmas divisões externas serão usadas por todos na comparação final.
cv_externa = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
METRICAS = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]

## 4. Busca e análise — ElasticNet

Combina L1 e L2. Valores de `l1_ratio` próximos de 1 dão maior peso à penalização L1.

Seguimos os mesmos passos: definir a grade, treinar a busca, examinar os candidatos e visualizar os coeficientes.

In [ ]:
# 1. Grade exclusiva do ElasticNet; as opções comuns são iguais nos três modelos.
param_grid_elasticnet = {
    "regressor__reg__alpha": ALPHAS,
    "regressor__reg__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9],
    "regressor__preprocessor": PREPROCESSADORES,
    "transformer": TRANSFORMADORES_ALVO,
}

# Mantemos o wrapper do alvo mesmo quando a grade escolhe transformer=None.
grid_search_elasticnet = grid_search_cv_regressor(
    regressor=ElasticNet(max_iter=100_000),
    param_grid=param_grid_elasticnet,
    preprocessor=preprocessamento_categoricas,
    target_transformer=target_transformer,
)

In [ ]:
# 2. Ajusta cada configuração por validação cruzada e reajusta a melhor em X, y.
grid_search_elasticnet.fit(X, y)

In [ ]:
# 3. Parâmetros escolhidos e RMSE médio da busca (positivo).
print("Melhores parâmetros:", grid_search_elasticnet.best_params_)
print(f"RMSE médio da busca: {-grid_search_elasticnet.best_score_:.3f}")
grid_search_elasticnet.best_estimator_

In [ ]:
# 4. Candidatos ordenados pelo RMSE; rank 1 indica a melhor configuração.
df_busca_elasticnet = pd.DataFrame(grid_search_elasticnet.cv_results_)
df_busca_elasticnet["MAE"] = -df_busca_elasticnet["mean_test_neg_mean_absolute_error"]
df_busca_elasticnet["RMSE"] = -df_busca_elasticnet["mean_test_neg_root_mean_squared_error"]
df_busca_elasticnet.sort_values("rank_test_neg_root_mean_squared_error")[[
    "params", "mean_test_r2", "MAE", "RMSE",
    "std_test_neg_root_mean_squared_error",
]].head(10)

In [ ]:
# 5. Coeficientes do modelo reajustado, com os nomes das variáveis transformadas.
melhor_elasticnet = grid_search_elasticnet.best_estimator_
pipeline_elasticnet = melhor_elasticnet.regressor_
df_coeficientes_elasticnet = dataframe_coeficientes(
    pipeline_elasticnet["reg"].coef_,
    pipeline_elasticnet["preprocessor"].get_feature_names_out(),
)
plot_coeficientes(df_coeficientes_elasticnet, tituto="Coeficientes — ElasticNet")
df_coeficientes_elasticnet

## 5. Como ler os resultados

- A busca escolhe o menor RMSE médio; `best_score_` guarda esse erro com sinal negativo.
- A tabela mostra R², MAE e RMSE dos candidatos; MAE e RMSE estão positivos.
- O gráfico mostra os coeficientes após o pré-processamento escolhido.
- Esses resultados foram usados para escolher parâmetros. A avaliação independente por validação cruzada aninhada está no notebook de comparação `02_GridSearchCV.ipynb`.
